# Algorithm 3.4 — State vector after a time $\Delta t$ (the capstone)

**Goal:** the payoff of all of Chapter 3. Given an initial state $(\mathbf{r}_0,\mathbf{v}_0)$ and an elapsed time $\Delta t$, return the new state $(\mathbf{r},\mathbf{v})$ — for any orbit, in one function.

**Code (answer key):** [`rv_from_r0v0`](../curtis/algorithms/alg_3_4_rv_from_r0v0.py) · **Book:** §3.7, Algorithm 3.4 · **Worked example:** 3.7

This notebook writes almost no new math — it **orchestrates** the pieces you already built: vis-viva → `kepler_U` → Lagrange $f,g$. The one new line is $\alpha$ from the energy equation.

## Read first

| Input | | Output | |
|---|---|---|---|
| $\mathbf{r}_0,\mathbf{v}_0$ | initial state | $\mathbf{r},\mathbf{v}$ | state after $\Delta t$ |
| $\Delta t$ | elapsed time (s) | | |

**The pipeline:**
$$(\mathbf r_0,\mathbf v_0)\ \xrightarrow{\text{vis-viva}}\ \alpha\ \xrightarrow{\texttt{kepler\_U}}\ \chi\ \xrightarrow{f,g,\dot f,\dot g}\ (\mathbf r,\mathbf v).$$

Pieces reused: `kepler_U` (3.3), `f_and_g` & `fDot_and_gDot` (Lagrange). New: $\alpha=2/r_0-v_0^2/\mu$.

## The picture (assembling the engine)

Everything in Chapter 3 was building toward this. Each earlier piece slots in:

1. **Vis-viva** turns the state's energy into $\alpha=1/a$ — the orbit's size, and the sign that says ellipse/parabola/hyperbola.
2. **`kepler_U`** turns the elapsed time $\Delta t$ into the universal anomaly $\chi$ (how far along the orbit you have gone).
3. **The Lagrange coefficients** turn $\chi$ into the four numbers that linearly map the old state to the new one.

No integration, no special-casing the conic — just composition. The figure below shows the result: the satellite carried along its orbit from where it started to where it is $\Delta t$ later.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, "..")
# Prerequisites you have already built:
from curtis.algorithms.alg_3_3_kepler_U import kepler_U
from curtis.algorithms.lagrange_coefficients import f_and_g
# Used only to draw the background orbit (from notebooks 4.1 / 4.2):
from curtis.algorithms.alg_4_1_coe_from_sv import coe_from_sv
from curtis.algorithms.alg_4_2_sv_from_coe import sv_from_coe

In [ ]:
# Example 3.7: propagate this state by dt = 3600 s and watch it move along the orbit.
mu = 398600.0
R0 = np.array([7000.0, -12124.0, 0.0])
V0 = np.array([2.6679, 4.6210, 0.0])
dt = 3600.0

# Background orbit: get its elements (4.1) and sweep true anomaly (4.2).
h, e, RA, incl, w, TA0, a = coe_from_sv(R0, V0, mu)
thetas = np.linspace(0, 2*np.pi, 400)
orbit = np.array([sv_from_coe([h, e, RA, incl, w, th], mu)[0] for th in thetas])

# End position after dt, via the prerequisites (kepler_U + Lagrange position half).
r0 = np.linalg.norm(R0); vr0 = np.dot(R0, V0)/r0
alpha = 2/r0 - np.linalg.norm(V0)**2/mu
x = kepler_U(dt, r0, vr0, alpha, mu)
f, g = f_and_g(x, dt, r0, alpha, mu)
Rend = f*R0 + g*V0

fig, ax = plt.subplots(figsize=(6.4, 6.2))
ax.plot(orbit[:, 0], orbit[:, 1], color='#73726c', lw=1.6)          # the orbit
ax.plot(0, 0, 'o', color='#3B8BD4', ms=9)                            # Earth (focus)
ax.plot(*R0[:2], 'o', color='#1D9E75', ms=7)
ax.plot(*Rend[:2], 'o', color='#D4537E', ms=7)
ax.plot([0, R0[0]], [0, R0[1]], color='#1D9E75', lw=1)
ax.plot([0, Rend[0]], [0, Rend[1]], color='#D4537E', lw=1)
ax.annotate('Earth', (0, 0), textcoords='offset points', xytext=(8, -12), color='#3B8BD4')
ax.annotate(r'start  $t=0$', R0[:2], textcoords='offset points', xytext=(8, -4), color='#1D9E75')
ax.annotate(r'after  $\Delta t=3600$ s', Rend[:2], textcoords='offset points', xytext=(8, 6), color='#D4537E')
ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Algorithm 3.4: propagate a state forward in time (Example 3.7)')
plt.show()

## See it — the satellite moved along its orbit

The grey curve is the orbit; Earth sits at the focus. In $\Delta t=3600$ s the satellite travelled from the green point to the pink one — roughly a fifth of the way around. That motion is the entire output of `rv_from_r0v0`: a new $\mathbf{r}$ (drawn) and a new $\mathbf{v}$ (tangent there, not shown).

Crucially, *nothing in the code knew or cared* that this orbit is an ellipse. Feed it a hyperbolic state and the identical pipeline — vis-viva, `kepler_U`, Lagrange — would carry the spacecraft along the open branch instead. That universality is what Chapter 3 was for.

## Where these equations come from

Only one equation here is new; the rest you have already derived.

### $\alpha$ from vis-viva (the one new line)
The specific energy $\varepsilon=\dfrac{v^2}{2}-\dfrac{\mu}{r}$ is constant and equals $-\dfrac{\mu}{2a}$. Setting these equal at the initial state and solving for $1/a$:
$$\alpha=\frac{1}{a}=\frac{2}{r_0}-\frac{v_0^2}{\mu}.$$
Its sign *is* the orbit type ($\alpha>0$ ellipse, $=0$ parabola, $<0$ hyperbola), which is exactly what `kepler_U` and the Stumpff functions need.

### The radial speed $v_{r0}$
`kepler_U` also needs the initial radial speed, $v_{r0}=\dfrac{\mathbf{r}_0\cdot\mathbf{v}_0}{r_0}$ — the component of velocity along $\mathbf{r}_0$.

### The rest is composition
$\chi=\texttt{kepler\_U}(\Delta t, r_0, v_{r0}, \alpha)$, then $f,g$ give $\mathbf{r}=f\mathbf{r}_0+g\mathbf{v}_0$, and $\dot f,\dot g$ (which need the *new* radius $r=|\mathbf{r}|$) give $\mathbf{v}=\dot f\mathbf{r}_0+\dot g\mathbf{v}_0$. Each was derived in its own notebook; 3.4 just runs them in order.

## Step by step (in code order)

1. **Magnitudes & radial speed:** $r_0=|\mathbf{r}_0|$, $v_0=|\mathbf{v}_0|$, $v_{r0}=\mathbf{r}_0\!\cdot\!\mathbf{v}_0/r_0$.
2. **Vis-viva:** $\alpha=2/r_0-v_0^2/\mu$.
3. **Universal anomaly:** $\chi=\texttt{kepler\_U}(\Delta t, r_0, v_{r0}, \alpha, \mu)$.
4. **Lagrange $f,g$:** `f, g = f_and_g(`$\chi, \Delta t, r_0, \alpha, \mu$`)`.
5. **New position:** $\mathbf{r}=f\mathbf{r}_0+g\mathbf{v}_0$, then $r=|\mathbf{r}|$.
6. **Lagrange $\dot f,\dot g$:** `fdot, gdot = fDot_and_gDot(`$\chi, r, r_0, \alpha, \mu$`)`.
7. **New velocity:** $\mathbf{v}=\dot f\mathbf{r}_0+\dot g\mathbf{v}_0$.

**↓ Now type your implementation below.** Pure orchestration — call the pieces in order. The only easy-to-miss detail: `fDot_and_gDot` needs the *new* radius `r`, so compute `R` (step 5) **before** step 6. Answer key linked above; peek only after you try.

In [ ]:
def rv_from_r0v0(R0, V0, t, mu):
    """State vector (R, V) at time t, propagated from the initial state (R0, V0)."""
    R0 = np.asarray(R0, dtype=float)
    V0 = np.asarray(V0, dtype=float)

    # 1. Magnitudes and initial radial speed:
    #        r0 = |R0|,  v0 = |V0|,  vr0 = (R0 . V0) / r0

    # 2. Reciprocal semimajor axis from vis-viva:  alpha = 2/r0 - v0**2/mu

    # 3. Universal anomaly:  x = kepler_U(t, r0, vr0, alpha, mu)

    # 4. f, g = f_and_g(x, t, r0, alpha, mu)

    # 5. R = f*R0 + g*V0 ;  r = |R|

    # 6. fdot, gdot = fDot_and_gDot(x, r, r0, alpha, mu)

    # 7. V = fdot*R0 + gdot*V0

    # return R, V
    raise NotImplementedError("fill in steps 1-7, then delete this line")

## Verify — Example 3.7

**Input** ($\mu_\oplus=398600$): $\;\mathbf{r}_0=[7000,-12124,0]$ km, $\;\mathbf{v}_0=[2.6679,4.6210,0]$ km/s, $\;\Delta t=3600$ s.

| | $x$ | $y$ | $z$ |
|---|---|---|---|
| $\mathbf{r}$ (km) | −3297.77 | 7413.40 | 0 |
| $\mathbf{v}$ (km/s) | −8.2976 | −0.964045 | 0 |

Run once your function is typed.

In [ ]:
mu = 398600.0
R0 = np.array([7000.0, -12124.0, 0.0])
V0 = np.array([2.6679, 4.6210, 0.0])
dt = 3600.0

R, V = rv_from_r0v0(R0, V0, dt, mu)

print(f"r (km)   = [{R[0]:10.6g} {R[1]:10.6g} {R[2]:10.6g}]   (expect [-3297.77  7413.40  0])")
print(f"v (km/s) = [{V[0]:10.6g} {V[1]:10.6g} {V[2]:10.6g}]   (expect [-8.2976  -0.964045  0])")

assert np.allclose(R, [-3297.77, 7413.40, 0.0], atol=1e-2)
assert np.allclose(V, [-8.2976, -0.964045, 0.0], atol=1e-4)
print("\nAll checks passed ✔")

## What this confirms

- **One propagator for every orbit.** vis-viva → `kepler_U` → Lagrange $f,g$ turns $(\mathbf r_0,\mathbf v_0,\Delta t)$ into $(\mathbf r,\mathbf v)$ with no conic-specific branches — the universal-variable payoff in full.
- The Chapter-3 chain is now **closed**: 3.1/3.2 (anomalies) → Stumpff → 3.3 (universal $\chi$) → Lagrange $f,g$ → 3.4 (state propagation). Each notebook fed the next.
- Together with 4.1/4.2 (state ⇄ elements) you can now move freely between *time*, *state vector*, and *orbital elements* — the core toolkit of two-body orbital mechanics.

**Next:** Chapter 5 — preliminary orbit determination, where you *recover* an orbit from observations: Gibbs' method (5.1, three position vectors) or Lambert's problem (5.2, two positions + time of flight).